# Automatic Differentiation with torch.autograd

When training neural networks, the most frequently used algorithm is back propagation. In this algorithm, parameters (model weights) are adjusted according to the gradient of the loss function with respect to the given parameter.

To compute those gradients, PyTorch has a built-in differentiation engine called torch.autograd. It supports automatic computation of gradient for any computational graph.

Consider the simplest one-layer neural network, with input x, parameters w and b, and some loss function. It can be defined in PyTorch in the following manner:


In [1]:
import torch

x = torch.ones(5)
y = torch.zeros(3)
w = torch.randn(5, 3, requires_grad=True)
b = torch.randn(3, requires_grad=True)
z = torch.matmul(x, w) + b
loss = torch.nn.functional.binary_cross_entropy_with_logits(z, y)


In this network, `w` and `b` are parameters, which we need to optimize. Thus, we need to be able to compute the gradients of loss function with respect to those variables. In order to do that, we set the `requires_grad` property of those tensors.

A function that we apply to tensors to construct computational graph is in fact an object of class Function. This object knows how to compute the function in the `forward` direction, and also how to compute its derivative during the `backward` propagation step. A reference to the backward propagation function is stored in `grad_fn` property of a tensor. 

In [2]:
print(f"Gradient function for z = {z.grad_fn}")
print(f"Gradient function for loss = {loss.grad_fn}")

Gradient function for z = <AddBackward0 object at 0x00000288CE6BADD0>
Gradient function for loss = <BinaryCrossEntropyWithLogitsBackward0 object at 0x00000288CE6B88E0>


## Computing Gradients
To optimize weights of parameters in the neural network, we need to compute the derivatives of our loss function with respect to parameters. To compute those derivatives, we call `loss.backward()`, and then retrieve the values from `w.grad` and `b.grad`:

In [3]:
loss.backward()
# grad: returns gradient for the Tensor
# grad_fn: returns the function that creates the Tensor
print(w.grad)
print(b.grad)

tensor([[0.0036, 0.1601, 0.0813],
        [0.0036, 0.1601, 0.0813],
        [0.0036, 0.1601, 0.0813],
        [0.0036, 0.1601, 0.0813],
        [0.0036, 0.1601, 0.0813]])
tensor([0.0036, 0.1601, 0.0813])


## Disabling Gradient Tracking
By default, all tensors with `requires_grad=True` are tracking their computational history and support gradient computation. However, there are some cases when we do not need to do that, for example, when we have trained the model and just want to apply it to some input data, i.e. we only want to do forward computations through the network. We can stop tracking computations by surrounding our computation code with `torch.no_grad()` 
block:

There are reasons you might want to disable gradient tracking:
- To mark some parameters in your neural network as frozen parameters.

- To speed up computations when you are only doing forward pass, because computations on tensors that do not track gradients would be more efficient.

In [4]:
z = torch.matmul(x, w) + b
print(z.requires_grad)

with torch.no_grad():
    z = torch.matmul(x, w) + b
print(z.requires_grad)

True
False


Another way to achieve the same result is to use the `detach()` method on the tensor:

In [ ]:
z = torch.matmul(x, w) + b
z_det = z.detach()
print(z_det.requires_grad)

False


`detach()` separates the tensor from its computation graph, without gradient spread, share the same memory.

`.clone().detach()` separates and create a new tensor in new memory.

`requires_grad=False` set up a new tensor which doesn't calculate the gradient.


In [11]:
x = torch.tensor([2.0, 3.0], requires_grad=True)
print("Original x: ", x)

y1 = x.detach()
y2 = x.clone().detach()
y3 = x.data
y4 = torch.tensor(x, requires_grad=False)

tensors = [("y1 (x.detach)", y1), 
           ("y2 (clone+detach)", y2), 
           ("y3 (x.data)", y3), 
           ("y4 (new tensor)", y4)]

print("\n=== Memory & requires_grad Comparison ===")
for name, t in tensors:
    print(f"{name:20} | requires_grad: {t.requires_grad} | id: {id(t)} | shares data with x: {t.data_ptr() == x.data_ptr()}")


Original x:  tensor([2., 3.], requires_grad=True)

=== Memory & requires_grad Comparison ===
y1 (x.detach)        | requires_grad: False | id: 2786625672592 | shares data with x: True
y2 (clone+detach)    | requires_grad: False | id: 2786625672672 | shares data with x: False
y3 (x.data)          | requires_grad: False | id: 2786625672912 | shares data with x: True
y4 (new tensor)      | requires_grad: False | id: 2786625672992 | shares data with x: False


C:\Users\zhang\AppData\Local\Temp\ipykernel_3780\3863298462.py:7: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  y4 = torch.tensor(x, requires_grad=False)


# More on Computatinal Graphs

Conceptually, autograd keeps a record of data (tensors) and all executed operations (along with the resulting new tensors) in a directed acyclic graph (DAG) consisting of Function objects. In this DAG, leaves are the input tensors, roots are the output tensors. By tracing this graph from roots to leaves, you can automatically compute the gradients using the chain rule.

In a forward pass, autograd does two things simultaneously:

- run the requested operation to compute a resulting tensor

- maintain the operation’s gradient function in the DAG.

The backward pass kicks off when .backward() is called on the DAG root. autograd then:

- computes the gradients from each .grad_fn,

- accumulates them in the respective tensor’s .grad attribute

- using the chain rule, propagates all the way to the leaf tensors.

In [12]:
inp = torch.eye(4, 5, requires_grad=True)
out = (inp+1).pow(2).t()

out.backward(torch.ones_like(out), retain_graph=True)
print(f"First call\n{inp.grad}")

out.backward(torch.ones_like(out), retain_graph=True)
print(f"\nSecond call\n{inp.grad}")

inp.grad.zero_()
out.backward(torch.ones_like(out), retain_graph=True)
print(f"\nCall after zeroing gradients\n{inp.grad}")

First call
tensor([[4., 2., 2., 2., 2.],
        [2., 4., 2., 2., 2.],
        [2., 2., 4., 2., 2.],
        [2., 2., 2., 4., 2.]])

Second call
tensor([[8., 4., 4., 4., 4.],
        [4., 8., 4., 4., 4.],
        [4., 4., 8., 4., 4.],
        [4., 4., 4., 8., 4.]])

Call after zeroing gradients
tensor([[4., 2., 2., 2., 2.],
        [2., 4., 2., 2., 2.],
        [2., 2., 4., 2., 2.],
        [2., 2., 2., 4., 2.]])
